# 06 — Corrected InceptionResNetV2 Frozen Baseline with Matched 13-Layer Head

**This is the only main-model rerun currently required.**

Purpose:
- keep the **entire InceptionResNetV2 ImageNet backbone frozen**
- use the same **13-layer classification-head sequence** as the fair baseline benchmark
- use `padding='same'` where needed so the 13-layer head fits the 5×5 InceptionResNetV2 output
- train on the **final leakage-controlled split**
- use the same controlled training schedule as the other final baseline models

## Fixed settings

- Dataset: `Data_Clean_LeakageControlled_FINAL`
- Seed: **42**
- Input: **224×224 RGB**
- Rescaling: **1/255**
- Batch size: **32**
- Training augmentation: horizontal + vertical flips only
- Optimizer: **Adam**
- Learning rate: **1e-4**
- Loss: **categorical cross-entropy**
- Epochs: **20**
- Best checkpoint: highest **validation accuracy**
- Test split remains untouched until the best validation checkpoint is selected

## Corrected 13-layer head

1. Conv2D(32, 3×3, padding='same')
2. BatchNormalization
3. MaxPooling2D(2×2, padding='same')
4. Dropout(0.17)
5. Conv2D(64, 2×2, padding='same')
6. BatchNormalization
7. GlobalAveragePooling2D
8. Dense(64)
9. Dense(32)
10. Dense(32)
11. BatchNormalization
12. Dropout(0.30)
13. Dense(3, Softmax)

The outputs are saved to a **new folder** and do not overwrite the previous InceptionResNetV2 run.

In [ ]:
# ============================================================
# CELL 1 — SETUP, DRIVE, GPU, PATHS
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os
import time
import json
import random
import platform
import gc

import numpy as np
import pandas as pd
import tensorflow as tf

from tensorflow.keras import layers, Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (
    ModelCheckpoint,
    CSVLogger,
    TerminateOnNaN
)
from tensorflow.keras.preprocessing.image import ImageDataGenerator

SEED = 42
EPOCHS = 20
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)
LEARNING_RATE = 1e-4

os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

MODEL_NAME = 'InceptionResNetV2_CORRECTED_13Layer_Frozen'

PROJECT = Path('/content/drive/MyDrive/Cataract')

DATA = PROJECT / 'Data_Clean_LeakageControlled_FINAL'
TRAIN_DIR = DATA / 'Train'
VAL_DIR = DATA / 'Validation'
TEST_DIR = DATA / 'Test'

OUT = (
    PROJECT
    / 'FINAL_REVISION_2026_08'
    / 'clean_split_models'
    / MODEL_NAME
)

OUT.mkdir(parents=True, exist_ok=True)

CLASS_ORDER = [
    'Cataract',
    'Normal',
    'Not Eye'
]

for p in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    assert p.exists(), f'STOP: Missing dataset folder: {p}'

gpus = tf.config.list_physical_devices('GPU')

print('Model:', MODEL_NAME)
print('TensorFlow:', tf.__version__)
print(
    'Keras:',
    tf.keras.__version__
    if hasattr(tf.keras, '__version__')
    else 'bundled'
)
print('Python:', platform.python_version())
print('GPU devices:', gpus)
print('Dataset:', DATA)
print('Output:', OUT)

if not gpus:
    raise RuntimeError(
        'STOP: No GPU detected. '
        'In Colab choose Runtime → Change runtime type → T4 GPU, '
        'then rerun CELL 1.'
    )

environment = {
    'model': MODEL_NAME,
    'architecture': 'InceptionResNetV2',
    'seed': SEED,
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'image_size': [224, 224],
    'learning_rate': LEARNING_RATE,
    'optimizer': 'Adam',
    'loss': 'categorical_crossentropy',
    'tensorflow': tf.__version__,
    'keras': (
        tf.keras.__version__
        if hasattr(tf.keras, '__version__')
        else 'bundled'
    ),
    'python': platform.python_version(),
    'gpu_devices': [str(x) for x in gpus],
    'dataset': str(DATA),
    'backbone_policy': 'fully_frozen',
    'head_layers': 13,
    'head_padding_note': (
        "padding='same' is used for both head convolutions "
        "and MaxPooling2D to preserve a valid spatial path "
        "from the 5x5 InceptionResNetV2 backbone output."
    )
}

with open(OUT / 'environment.json', 'w') as f:
    json.dump(environment, f, indent=2)

print('\n✅ CELL 1 COMPLETE')

In [ ]:
# ============================================================
# CELL 2 — DATA GENERATORS + COUNT CHECK
# ============================================================

train_aug = ImageDataGenerator(
    rescale=1./255,
    horizontal_flip=True,
    vertical_flip=True
)

plain = ImageDataGenerator(
    rescale=1./255
)

train = train_aug.flow_from_directory(
    TRAIN_DIR,
    target_size=IMAGE_SIZE,
    color_mode='rgb',
    class_mode='categorical',
    classes=CLASS_ORDER,
    batch_size=BATCH_SIZE,
    shuffle=True,
    seed=SEED,
    interpolation='nearest'
)

val = plain.flow_from_directory(
    VAL_DIR,
    target_size=IMAGE_SIZE,
    color_mode='rgb',
    class_mode='categorical',
    classes=CLASS_ORDER,
    batch_size=BATCH_SIZE,
    shuffle=False,
    interpolation='nearest'
)

test = plain.flow_from_directory(
    TEST_DIR,
    target_size=IMAGE_SIZE,
    color_mode='rgb',
    class_mode='categorical',
    classes=CLASS_ORDER,
    batch_size=BATCH_SIZE,
    shuffle=False,
    interpolation='nearest'
)

print('\nClass indices:', train.class_indices)
print('Train images:', train.samples)
print('Validation images:', val.samples)
print('Test images:', test.samples)

assert train.samples == 8845, (
    f'Unexpected Train count: {train.samples}'
)

assert val.samples == 2178, (
    f'Unexpected Validation count: {val.samples}'
)

assert test.samples == 2587, (
    f'Unexpected Test count: {test.samples}'
)

assert train.class_indices == {
    'Cataract': 0,
    'Normal': 1,
    'Not Eye': 2
}

print('\n✅ FINAL CLEAN DATASET COUNTS VERIFIED')
print('✅ CELL 2 COMPLETE')

In [ ]:
# ============================================================
# CELL 3 — BUILD CORRECTED FROZEN INCEPTIONRESNETV2
# ============================================================

tf.keras.backend.clear_session()
gc.collect()

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

base = tf.keras.applications.InceptionResNetV2(
    weights='imagenet',
    include_top=False,
    input_shape=(224, 224, 3)
)

print('Backbone output shape:', base.output_shape)
print('Backbone layers:', len(base.layers))

# Entire pretrained backbone must be frozen.
for layer in base.layers:
    layer.trainable = False

x = base.output

# ------------------------------------------------------------
# Corrected common 13-layer head
# ------------------------------------------------------------

# 1
x = layers.Conv2D(
    32,
    (3, 3),
    activation='relu',
    padding='same',
    name='head_conv32'
)(x)

# 2
x = layers.BatchNormalization(
    name='head_bn1'
)(x)

# 3
x = layers.MaxPooling2D(
    (2, 2),
    padding='same',
    name='head_pool'
)(x)

# 4
x = layers.Dropout(
    0.17,
    name='head_dropout1'
)(x)

# 5
x = layers.Conv2D(
    64,
    (2, 2),
    activation='relu',
    padding='same',
    name='head_conv64'
)(x)

# 6
x = layers.BatchNormalization(
    name='head_bn2'
)(x)

# 7
x = layers.GlobalAveragePooling2D(
    name='head_gap'
)(x)

# 8
x = layers.Dense(
    64,
    activation='relu',
    name='head_dense64'
)(x)

# 9
x = layers.Dense(
    32,
    activation='relu',
    name='head_dense32a'
)(x)

# 10
x = layers.Dense(
    32,
    activation='relu',
    name='head_dense32b'
)(x)

# 11
x = layers.BatchNormalization(
    name='head_bn3'
)(x)

# 12
x = layers.Dropout(
    0.30,
    name='head_dropout2'
)(x)

# 13
output = layers.Dense(
    3,
    activation='softmax',
    name='preds'
)(x)

model = Model(
    inputs=base.input,
    outputs=output,
    name='InceptionResNetV2_Corrected13Layer'
)

model.compile(
    optimizer=Adam(
        learning_rate=LEARNING_RATE
    ),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# ------------------------------------------------------------
# Audit trainability
# ------------------------------------------------------------

backbone_layer_names = {
    layer.name
    for layer in base.layers
}

trainability = pd.DataFrame([
    {
        'index': i,
        'layer': layer.name,
        'class': layer.__class__.__name__,
        'trainable': bool(layer.trainable),
        'in_backbone': layer.name in backbone_layer_names
    }
    for i, layer in enumerate(model.layers)
])

trainability.to_csv(
    OUT / 'layer_trainability.csv',
    index=False
)

backbone_trainable = [
    layer.name
    for layer in base.layers
    if layer.trainable
]

head_layers = [
    layer
    for layer in model.layers
    if layer.name not in backbone_layer_names
]

frozen_head = [
    layer.name
    for layer in head_layers
    if not layer.trainable
]

print('\nTotal saved model layers:', len(model.layers))
print('Backbone layers:', len(base.layers))
print('Custom head layers:', len(head_layers))
print('Trainable backbone layers:', len(backbone_trainable))
print('Frozen head layers:', len(frozen_head))

print('\nHead sequence:')
for i, layer in enumerate(head_layers, start=1):
    print(
        f'{i:02d}.',
        layer.name,
        '|',
        layer.__class__.__name__,
        '| trainable =',
        layer.trainable
    )

# Critical scientific assertions
assert len(base.layers) == 780, (
    f'Expected 780 InceptionResNetV2 backbone layers, '
    f'found {len(base.layers)}'
)

assert len(backbone_trainable) == 0, (
    'STOP: At least one pretrained backbone layer is trainable.'
)

assert len(head_layers) == 13, (
    f'STOP: Corrected head must contain 13 layers, '
    f'found {len(head_layers)}'
)

assert len(frozen_head) == 0, (
    'STOP: At least one custom head layer is frozen.'
)

head_class_sequence = [
    layer.__class__.__name__
    for layer in head_layers
]

expected_head_classes = [
    'Conv2D',
    'BatchNormalization',
    'MaxPooling2D',
    'Dropout',
    'Conv2D',
    'BatchNormalization',
    'GlobalAveragePooling2D',
    'Dense',
    'Dense',
    'Dense',
    'BatchNormalization',
    'Dropout',
    'Dense'
]

assert head_class_sequence == expected_head_classes, (
    'STOP: Head class sequence does not match the intended 13-layer head.'
)

print('\n✅ ENTIRE BACKBONE FROZEN')
print('✅ ALL 13 CUSTOM HEAD LAYERS TRAINABLE')
print('✅ CORRECTED HEAD STRUCTURE VERIFIED')
print('✅ CELL 3 COMPLETE')

In [ ]:
# ============================================================
# CELL 4 — TRAIN 20 EPOCHS
# ============================================================

DONE = OUT / 'DONE.txt'
BEST = OUT / 'best.keras'

if DONE.exists():
    raise RuntimeError(
        f'STOP: Corrected Inception run is already complete: {DONE}. '
        'Do not retrain it.'
    )

callbacks = [
    ModelCheckpoint(
        BEST,
        monitor='val_accuracy',
        mode='max',
        save_best_only=True,
        verbose=1
    ),
    CSVLogger(
        OUT / 'history.csv'
    ),
    TerminateOnNaN()
]

print('Starting corrected InceptionResNetV2 training')
print('Epochs:', EPOCHS)
print('Seed:', SEED)
print('Best checkpoint:', BEST)

t0 = time.time()

history = model.fit(
    train,
    validation_data=val,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

training_seconds = time.time() - t0

(
    OUT
    / 'training_time_seconds.txt'
).write_text(
    str(training_seconds)
)

print(
    '\nTraining time (minutes):',
    training_seconds / 60
)

print(
    '✅ CELL 4 COMPLETE — training finished '
    'and best validation checkpoint saved.'
)

In [ ]:
# ============================================================
# CELL 5 — VERIFY BEST EPOCH + SAVE TRAINING SUMMARY
# ============================================================

history_df = pd.read_csv(
    OUT / 'history.csv'
)

assert 'val_accuracy' in history_df.columns

best_idx = int(
    history_df[
        'val_accuracy'
    ].astype(float).idxmax()
)

best_epoch = best_idx + 1

best_val_accuracy = float(
    history_df.loc[
        best_idx,
        'val_accuracy'
    ]
)

training_summary = {
    'model': MODEL_NAME,
    'seed': SEED,
    'epochs_completed': len(history_df),
    'best_epoch': best_epoch,
    'best_validation_accuracy': best_val_accuracy,
    'training_seconds': float(training_seconds),
    'training_minutes': float(training_seconds / 60)
}

with open(
    OUT / 'training_summary.json',
    'w'
) as f:
    json.dump(
        training_summary,
        f,
        indent=2
    )

print('Epochs completed:', len(history_df))
print('Best epoch:', best_epoch)
print('Best validation accuracy:', best_val_accuracy)

assert len(history_df) == 20, (
    f'Expected 20 history rows, found {len(history_df)}'
)

assert BEST.exists(), (
    'STOP: best.keras was not saved.'
)

print('\n✅ CELL 5 COMPLETE')

In [ ]:
# ============================================================
# CELL 6 — TEST PREDICTIONS FROM BEST VALIDATION CHECKPOINT
# ============================================================

best_model = tf.keras.models.load_model(
    BEST,
    compile=False
)

test.reset()

probs = best_model.predict(
    test,
    verbose=1
)

y_true = test.classes.copy()
y_pred = probs.argmax(axis=1)

assert probs.shape == (2587, 3)
assert y_true.shape == (2587,)
assert y_pred.shape == (2587,)

assert np.isfinite(probs).all()

assert np.allclose(
    probs.sum(axis=1),
    1.0,
    atol=1e-3
)

np.save(
    OUT / 'probs.npy',
    probs
)

np.save(
    OUT / 'y_true.npy',
    y_true
)

np.save(
    OUT / 'y_pred.npy',
    y_pred
)

pd.DataFrame({
    'filepath': test.filepaths,
    'y_true': y_true,
    'y_pred': y_pred
}).to_csv(
    OUT / 'test_predictions_index.csv',
    index=False
)

print('Prediction array shape:', probs.shape)
print('y_true shape:', y_true.shape)
print('y_pred shape:', y_pred.shape)

print('\n✅ CELL 6 COMPLETE')

In [ ]:
# ============================================================
# CELL 7 — QUICK FINAL METRICS
# ============================================================

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    confusion_matrix
)

# ------------------------------------------------------------
# 3-class metrics
# ------------------------------------------------------------

test_accuracy = accuracy_score(
    y_true,
    y_pred
)

(
    precision,
    recall,
    f1,
    support
) = precision_recall_fscore_support(
    y_true,
    y_pred,
    labels=[0, 1, 2],
    zero_division=0
)

macro_precision = float(
    np.mean(precision)
)

macro_recall = float(
    np.mean(recall)
)

macro_f1 = float(
    np.mean(f1)
)

macro_auc = roc_auc_score(
    y_true,
    probs,
    multi_class='ovr',
    average='macro'
)

cm3 = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1, 2]
)

# ------------------------------------------------------------
# Clinical Cataract-v-Normal metrics
# ------------------------------------------------------------

clinical_mask = np.isin(
    y_true,
    [0, 1]
)

clinical_true3 = y_true[
    clinical_mask
]

clinical_probs = probs[
    clinical_mask
]

y_binary = (
    clinical_true3 == 0
).astype(int)

denominator = (
    clinical_probs[:, 0]
    + clinical_probs[:, 1]
)

cataract_score = np.divide(
    clinical_probs[:, 0],
    denominator,
    out=np.full(
        denominator.shape,
        0.5,
        dtype=float
    ),
    where=denominator > 0
)

binary_pred = (
    cataract_score >= 0.5
).astype(int)

TP = int(
    (
        (y_binary == 1)
        &
        (binary_pred == 1)
    ).sum()
)

FN = int(
    (
        (y_binary == 1)
        &
        (binary_pred == 0)
    ).sum()
)

TN = int(
    (
        (y_binary == 0)
        &
        (binary_pred == 0)
    ).sum()
)

FP = int(
    (
        (y_binary == 0)
        &
        (binary_pred == 1)
    ).sum()
)

clinical_accuracy = (
    TP + TN
) / len(y_binary)

sensitivity = (
    TP
    / (TP + FN)
)

specificity = (
    TN
    / (TN + FP)
)

clinical_auc = roc_auc_score(
    y_binary,
    cataract_score
)

summary = {
    'Model': MODEL_NAME,
    'Test_N': int(len(y_true)),
    'Test_3Class_Accuracy': float(test_accuracy),
    'Macro_Precision': macro_precision,
    'Macro_Recall': macro_recall,
    'Macro_F1': macro_f1,
    'Macro_AUC_OVR': float(macro_auc),
    'Clinical_N': int(len(y_binary)),
    'Cataract_N': int(y_binary.sum()),
    'Normal_N': int((y_binary == 0).sum()),
    'Clinical_Accuracy': float(clinical_accuracy),
    'Sensitivity': float(sensitivity),
    'Specificity': float(specificity),
    'Clinical_AUC': float(clinical_auc),
    'TP': TP,
    'FN': FN,
    'TN': TN,
    'FP': FP,
    'Best_Epoch': int(best_epoch),
    'Best_Validation_Accuracy': float(best_val_accuracy),
    'Training_Seconds': float(training_seconds)
}

with open(
    OUT / 'final_summary.json',
    'w'
) as f:
    json.dump(
        summary,
        f,
        indent=2
    )

pd.DataFrame(
    [summary]
).to_csv(
    OUT / 'final_summary.csv',
    index=False
)

pd.DataFrame(
    cm3,
    index=CLASS_ORDER,
    columns=CLASS_ORDER
).to_csv(
    OUT / 'confusion_matrix_3class.csv'
)

print('\n========================================')
print('CORRECTED INCEPTION FINAL RESULTS')
print('========================================')

print(
    '3-class Test Accuracy:',
    f'{100 * test_accuracy:.4f}%'
)

print(
    'Clinical Accuracy:',
    f'{100 * clinical_accuracy:.4f}%'
)

print(
    'Sensitivity:',
    f'{100 * sensitivity:.4f}%'
)

print(
    'Specificity:',
    f'{100 * specificity:.4f}%'
)

print(
    'Clinical AUC:',
    f'{clinical_auc:.6f}'
)

print(
    'TP / FN / TN / FP:',
    TP, FN, TN, FP
)

print('\n3-class confusion matrix:')
print(cm3)

print('\n✅ CELL 7 COMPLETE')

In [ ]:
# ============================================================
# CELL 8 — FINAL INTEGRITY CHECK + DONE MARKER
# ============================================================

required_outputs = [
    'best.keras',
    'history.csv',
    'training_time_seconds.txt',
    'environment.json',
    'layer_trainability.csv',
    'training_summary.json',
    'probs.npy',
    'y_true.npy',
    'y_pred.npy',
    'test_predictions_index.csv',
    'final_summary.json',
    'final_summary.csv',
    'confusion_matrix_3class.csv'
]

missing = []

for fn in required_outputs:

    p = OUT / fn

    if (
        not p.exists()
        or p.stat().st_size == 0
    ):
        missing.append(fn)

if missing:

    print('Missing or empty outputs:')

    for fn in missing:
        print('❌', fn)

    raise RuntimeError(
        'STOP: Corrected Inception run is incomplete.'
    )

DONE.write_text(
    'completed\n'
    'architecture=InceptionResNetV2\n'
    'configuration=fully frozen backbone + corrected matched 13-layer head\n'
    'seed=42\n'
    'epochs=20\n'
)

print('========================================')
print('✅ CORRECTED INCEPTIONRESNETV2 COMPLETE')
print('========================================')

print('\nOutput folder:')
print(OUT)

print(
    '\nNEXT: Download/save this executed notebook '
    'and upload it back to ChatGPT.'
)

print(
    '\nDO NOT START ANOTHER MODEL TRAINING RUN.'
)